In [14]:
import sys

sys.path.append('../../../')

In [ ]:
import types


In [2]:
from __future__ import annotations
from pathlib import Path
from typing import Any

import re
import yaml

import torch
import numpy as np

from computer_vision.yolov11_pose.nn.tasks import cfg2task

class Model(torch.nn.Module):
    """
    A base class for implementing YOLO models.
    This class provides a common interface for various operations related to YOLO models, such as training, validation, prediction.
    """
    def __init__(self, model:str|Path,task:str|None=None, verbose:bool=False)->None:
        """
        Initialize a new instance of YOLO
        Args:
            model (str|Path): Path or name of the model to load or create
            task (str, optional): The specific task for the model. If None, it will be inferred from the config
            verbose (bool): If True, enable verbose output 
        """
        super().__init__()
        self.predictor=None 
        self.model=None
        self.trainer=None
        self.cfg=None # if loaded from *.yaml
        self.ckpt={} # if loaded from *.pt
        self.ckpt_path=None
        self.overrides={} # overrides for trainer object
        self.metrics = None # validation/training metrics
        self.task=task # task type
        self.model_name=None # model name
        model=str(model).strip()
        # Load or create new YOLO model
        __import__("os").environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # to avoid deterministic warnings
        # if str(model).endswith((".yaml", ".yml")): self._new(model, task=task,verbose=verbose)
        # else: self._load(model, task=task)

        # # Delete super().training for accessing self.model.training
        # del self.training

    @property
    def task_map(self)->dict[str, dict[str, Any]]:
        """
        Map head to model, trainer, validator, and predictor classes
        """
        return {"pose":{"model":PoseModel}}
        #return {"pose":{"model":PoseModel,"trainer":yolo.pose.PoseTrainer, "validator":yolo.pose.PoseValidator, "predictor":yolo.pose.PosePredictor}}

    def _new(self, cfg:str, task=None, model=None,verbose=False)->None:
        """
        Initialize a new model and infer the task type from model definition
        Create a new model instance based on the provided configuration file. Load the model configuration, infer the task type if not specified,
        and initialize the model using the appropriate class from the task map
        Args:
            cfg (str): Path to the model configuration file in YAML format
            task (str, optional): The specific task for the model. If None, it will be inferred from the config
            model (torch.nn.Module, optional): A custom model instance. If provided, it will be used instead of creating a new one
            verbose (bool): If True, display model information during loading
        """
        self.cfg=cfg
        if isinstance(cfg, str): cfg=Path(cfg)
        if isinstance(cfg, Path):
            assert cfg.is_file(), f'{cfg} does not exist'
            with open(cfg) as f: cfg_dict=yaml.load(f, Loader=yaml.SafeLoader)
        elif not isinstance(cfg, dict): raise TypeError(f'cfg must be dict/str but got {type(cfg)}')
        else: cfg_dict=cfg

        # guess model scale
        try: cfg_dict['scale']=re.search(r"yolo(e-)?[v]?\d+([nslmx])", cfg.stem).group(2)
        except AttributeError: cfg_dict['scale']=''
        cfg_dict["yaml_file"] = str(cfg)
        # guess task 
        self.task=cfg2task(cfg_dict)

cfg='../yolo11-pose.yaml'
model=Model(model=cfg, verbose=True)

In [18]:
cfg_dict["nc"]

80

'pose'